<a href="https://colab.research.google.com/github/dbonjd007/ds-core-projects/blob/main/recommendation-systems/ContentBaseRecommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

In [ ]:
tfidf = TfidfVectorizer(token_pattern=r'[^|]+')
tfidf_matrix = tfidf.fit_transform(movies["genres"])

In [ ]:
genre_similarity = cosine_similarity(tfidf_matrix)

In [ ]:
user_movie_matrix = ratings.pivot_table(index="userId", columns="movieId",values="rating")
print(user_movie_matrix)

movieId  1       2       3       4       ...  193583  193585  193587  193609
userId                                   ...                                
1           4.0     NaN     4.0     NaN  ...     NaN     NaN     NaN     NaN
2           NaN     NaN     NaN     NaN  ...     NaN     NaN     NaN     NaN
3           NaN     NaN     NaN     NaN  ...     NaN     NaN     NaN     NaN
4           NaN     NaN     NaN     NaN  ...     NaN     NaN     NaN     NaN
5           4.0     NaN     NaN     NaN  ...     NaN     NaN     NaN     NaN
...         ...     ...     ...     ...  ...     ...     ...     ...     ...
606         2.5     NaN     NaN     NaN  ...     NaN     NaN     NaN     NaN
607         4.0     NaN     NaN     NaN  ...     NaN     NaN     NaN     NaN
608         2.5     2.0     2.0     NaN  ...     NaN     NaN     NaN     NaN
609         3.0     NaN     NaN     NaN  ...     NaN     NaN     NaN     NaN
610         5.0     NaN     NaN     NaN  ...     NaN     NaN     NaN     NaN

In [ ]:
def hybrid_recommend(user_id, movie_id, top_n = 5, alpha = 0.5):
  if movie_id not in movies ['movieId'].values:
    return "Movie not found."

  # Find similar movies
  movie_index = movies[movies['movieId'] == movie_id].index[0]
  similar_index = genre_similarity[movie_index].argsort()[::-1]

  # Prepare hybrid score
  recommendations = []
  for sim_idx in similar_index[1:]:
    similar_movie_id = movies.iloc[sim_idx]["movieId"]
    if similar_movie_id not in user_movie_matrix.columns:
      continue

    genre_similarity_score = genre_similarity[movie_index][sim_idx]
    user_movie_rating = user_movie_matrix[similar_movie_id].mean()/5
    hybrid_score = alpha * genre_similarity_score + (1-alpha) * user_movie_rating
    recommendations.append((movies.iloc[sim_idx]["title"], round(hybrid_score,2)))

    if len(recommendations) >= top_n:
      break
  return pd.DataFrame(recommendations, columns=["Movies","Hybrid Scores"])

In [ ]:
print(hybrid_recommend(1,1))

                                           Movies  Hybrid Scores
0                                    Turbo (2013)           0.75
1                Emperor's New Groove, The (2000)           0.87
2  Adventures of Rocky and Bullwinkle, The (2000)           0.72
3                              Toy Story 2 (1999)           0.89
4                  Tale of Despereaux, The (2008)           0.80
